<a href="https://colab.research.google.com/github/Jimber120/DA-Project-Member-3/blob/main/work/notebooks/01_problem_framing_and_data_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Problem Framing, Hypotheses & Data Ingestion: Ranking Signal Analysis

**Project:** FlyRank Ranking Signal Analysis — Predicting Continuous Search Visibility  
**Pipeline Stage:** Notebook 01 of 03 (Problem Framing, Hypotheses & Feature Vector Assembly)  
**Target Variable:** Continuous Search Visibility $\log(1 + \text{impressions\_90d})$ on Held-Out Enterprise Client Portfolios  
**Standardized Split:** Grouped Holdout (`GroupShuffleSplit`, `test_size=0.20`, `random_state=42`) with Zero Client Domain Overlap  

---

### Notebook Architecture & Objectives
This notebook is your starting template for the data audit and feature engineering phase:
1. **Problem Framing & Operational Decision:** Define the operational content refresh decision, continuous search visibility target, and asymmetric error trade-offs.
2. **Core SEO Hypotheses:** Formulate and statistically test 4 core search hypotheses with explicit empirical verdicts (`CONFIRMED`, `MIXED`, `FALSE`, `OPPOSITE`).
3. **Data Ingestion & Scope Audit:** Verify atomic entity grain (`content_id`), document scope limitations, missingness patterns, and rate scaling.
4. **Feature Vector Assembly & Leakage Defense:** Implement SERP inverse power-law ranking signals, engagement ratios, missingness indicators, log-stabilizing transforms, enforce strict leakage blacklists, and export the validated feature dataset.

## 1. Problem Framing & Operational Decision

### 1.1 The Operational Decision
* **What operational decision does this model improve?**  
  Enterprise content and SEO teams manage portfolios spanning thousands of URLs across diverse client domains. Currently, content refresh prioritization relies on crude, lagging manual heuristics (e.g., sorting by calendar age or raw pageviews). This model predicts expected **search visibility** (the latent search impression capacity of a page based on its structural, semantic, and engagement signals). Content strategists and engineering teams act on this model to identify:
  1. **Under-indexed high-potential pages:** Content whose observed features warrant high search impressions but currently lag behind their predicted potential (prime refresh/optimization candidates).
  2. **Over-resourced low-potential pages:** Thin or poorly positioned content that should not consume scarce editorial rewrite hours.

* **Who acts on it?**  
  Content operations leads, technical SEO specialists, and client account managers.

### 1.2 Target Definition: Search Visibility
* **Why Continuous Regression instead of Binary Classification?**  
  Binary classification collapses search performance into an arbitrary threshold (e.g., "declining" vs. "stable"), obscuring orders of magnitude of search impact. Continuous regression models the true scale of visibility across the entire distribution.
* **Target Formulation:**
  $$\text{target\_log\_impressions} = \log(1 + \text{impressions\_90d})$$
  Raw search impressions follow an extreme power-law distribution spanning from single digits to hundreds of thousands. The $\log(1+x)$ transformation compresses extreme right-tail skewness into a well-behaved, symmetric distribution suitable for regression estimators.

### 1.3 Asymmetric Cost of Errors
* **Overestimation Error (False High Visibility):** The model predicts high visibility for low-yield content.  
  *Operational Cost:* Editorial teams waste expensive subject-matter expert hours rewriting, optimizing, and re-indexing pages that fail to generate organic traffic. Opportunity cost is high.
* **Underestimation Error (False Low Visibility):** The model predicts low visibility for high-yield content.  
  *Operational Cost:* The team neglects high-leverage assets, risking rank decay against aggressive competitor content.

## 2. Core SEO Hypotheses

Before building machine learning models, formulate four operational hypotheses based on prevailing SEO heuristics to test whether real-world data supports common industry beliefs:
1. **Hypothesis 1 (Content Depth):** *Article length (`word_count`) exhibits a strong positive relationship with search visibility.*
2. **Hypothesis 2 (Search Demand):** *Target keyword search volume (`search_volume`) strongly determines realized page impressions.*
3. **Hypothesis 3 (SERP Positioning):** *Average SERP rank position (`avg_position`) is inversely correlated with impressions (lower numerical rank = higher visibility).*
4. **Hypothesis 4 (Content Freshness):** *Recently updated content (`days_since_last_update` lower) commands higher search visibility than aged content.*

Each hypothesis should be evaluated using Spearman rank correlation ($\rho$), Pearson correlation ($r$), and visual distribution inspections, concluding with an explicit statistical verdict (`CONFIRMED`, `OPPOSITE`, `MIXED`, or `FALSE`).

In [6]:
# Code Cell 1: Setup, Imports, and Data Ingestion
# ----------------------------------------------------------------------
# 1a. Imports
# ----------------------------------------------------------------------
import os, json, warnings
import numpy as np
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
RANDOM_STATE = 42

# ----------------------------------------------------------------------
# 1b. Locate project root (folder containing data/raw/...) wherever the notebook is launched from
# ----------------------------------------------------------------------
RAW_REL = os.path.join("data", "raw", "content_refresh_anonymized.csv")
ROOT = os.getcwd()
while not os.path.exists(os.path.join(ROOT, RAW_REL)):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        raise FileNotFoundError(f"Could not find {RAW_REL} above {os.getcwd()}")
    ROOT = parent
RAW_PATH = os.path.join(ROOT, RAW_REL)
OUT_DIR = os.path.join(ROOT, "work", "outputs")
os.makedirs(OUT_DIR, exist_ok=True)
print("Project root:", ROOT)

# ----------------------------------------------------------------------
# 1c. Ingest + entity-grain assertions
# ----------------------------------------------------------------------
df_raw = pd.read_csv(RAW_PATH)
assert df_raw["content_id"].is_unique, "Primary key content_id is not unique!"
assert df_raw["client_id"].nunique() == 32, f"Expected 32 clients, found {df_raw['client_id'].nunique()}"
print(f"Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns (expected 30,000 x 44)")
print(f"Clients: {df_raw['client_id'].nunique()}")

# ----------------------------------------------------------------------
# 1d. Continuous target: log(1 + impressions_90d)
# ----------------------------------------------------------------------
TARGET = "target_log_impressions"
df = df_raw.copy()
df[TARGET] = np.log1p(df["impressions_90d"])

print("\nTarget summary")
print(df[TARGET].describe().round(4).to_string())
print(f"\nSkew: raw impressions_90d = {df['impressions_90d'].skew():.2f}  ->  log1p target = {df[TARGET].skew():.2f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
sns.histplot(df["impressions_90d"].clip(upper=df["impressions_90d"].quantile(0.99)), bins=60, ax=ax[0])
ax[0].set_title("impressions_90d (clipped at p99)")
sns.histplot(df[TARGET], bins=60, kde=True, ax=ax[1], color="darkorange")
ax[1].set_title(TARGET)
plt.tight_layout(); plt.show()

FileNotFoundError: Could not find data/raw/content_refresh_anonymized.csv above /content

## 3. Data Ingestion: Scope & Limitations

### 3.1 Scope of the Dataset
* **Dataset File:** `data/raw/content_refresh_anonymized.csv` (30,000 rows × 44 columns).
* **Entity Grain:** Exactly 1 row per unique pseudonymized content item (`content_id`).
* **Client Representation:** Exactly 32 distinct pseudonymized enterprise client domains (`client_id`).
* **Observation Window:** A fixed trailing 90-day observation window ending at export time, with 30-day comparative windows (`*_last_30d` vs `*_prev_30d`).
* **Cohort Age:** Established, mature content only (`content_age_days >= 90`). Every row has at least 1 impression (`impressions_90d >= 1`).

### 3.2 Key Data Gotchas & Limitations
1. **Rate Scaling (×100):** Columns `ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, and `trend_pct` are recorded as ×100 percentages (`0.76` means `0.76%`, not `76%`).
2. **`avg_position = 0` Unranked State:** 1,205 rows have `avg_position = 0`, denoting unranked status / no impression position recorded, not rank zero.
3. **Systematic Missingness:** Keyword context is missing along `content_type` lines (e.g. `feedly article` rows have no keyword data; ~25.7% of rows lack `word_count`). Blind `fillna(0)` injects artificial category signals; missingness indicators are required.
4. **Observable Telemetry Only:** Internal product decision flags (`health_score`, `priority_score`, `action_type`) are deliberately excluded to prevent circular result leakage.

In [ ]:
# Code Cell 2: Completeness and Missingness Audit
# ----------------------------------------------------------------------
# 2a. Missing values: counts and percentages
# ----------------------------------------------------------------------
miss = pd.DataFrame({"n_missing": df_raw.isna().sum(),
                     "pct_missing": 100 * df_raw.isna().mean()})
miss = miss[miss["n_missing"] > 0].sort_values("pct_missing", ascending=False)
print("Columns with missing data:")
display(miss)

# ----------------------------------------------------------------------
# 2b. Zero-value audit on numeric features (zeros are often 'unknown', not 'none')
# ----------------------------------------------------------------------
num = df_raw.select_dtypes("number")
zeros = pd.DataFrame({"n_zero": (num == 0).sum(), "pct_zero": 100 * (num == 0).mean()})
zeros = zeros[zeros["n_zero"] > 0].sort_values("pct_zero", ascending=False)
print("\nNumeric columns containing zeros:")
display(zeros.head(25))

# ----------------------------------------------------------------------
# 2c. Known gotchas from the data contract
# ----------------------------------------------------------------------
print(f"\navg_position == 0 (unranked / no position recorded): {(df_raw['avg_position'] == 0).sum():,} rows")
print(f"search_volume == 0 : {(df_raw['search_volume'] == 0).sum():,} rows")
print("Scope: content_age_days min =", df_raw["content_age_days"].min(), "(expected >= 90)")
print("Scope: impressions_90d  min =", df_raw["impressions_90d"].min(), "(expected >= 1)")
print("=> Findings apply to mature, already-impressed pages only; not to brand-new or zero-impression URLs.")

# Missingness by content_type is systematic, not random
if "content_type" in df_raw.columns:
    by_type = (df_raw.assign(**{c: df_raw[c].isna() for c in miss.index[:4]})
                     .groupby("content_type")[list(miss.index[:4])].mean().mul(100).round(1))
    print("\n% missing by content_type (top-4 missing columns):")
    display(by_type)

In [ ]:
# Code Cell 3: Statistical Hypothesis Testing with Explicit Verdicts
# ----------------------------------------------------------------------
# 3a. Correlation helper (Spearman rho + Pearson r, with p-values)
# ----------------------------------------------------------------------
def corr_report(x, y):
    m = x.notna() & y.notna()
    x, y = x[m], y[m]
    rho, p_rho = st.spearmanr(x, y)
    r, p_r = st.pearsonr(x, y)
    return rho, p_rho, r, p_r, len(x)

# Verdict rules (documented so they are reproducible):
#   |rho| >= 0.10 AND significant, expected direction  -> CONFIRMED
#   |rho| >= 0.10 AND significant, opposite direction  -> OPPOSITE
#   |rho| <  0.10 but significant, opposite direction  -> MIXED  (a real but weak/contradicting signal)
#   otherwise (too weak / not significant)             -> FALSE  (belief not supported)
def verdict(rho, p, expected_sign):
    if p >= 0.05:
        return "FALSE"
    if abs(rho) >= 0.10:
        return "CONFIRMED" if np.sign(rho) == expected_sign else "OPPOSITE"
    return "FALSE" if np.sign(rho) == expected_sign else "MIXED"

df["log_search_volume_tmp"] = np.log1p(df["search_volume"])
ranked = df[df["avg_position"] > 0]

# expected_sign: +1 means 'higher x => higher visibility' is the industry belief.
# H3: LOWER position number (better rank) => more visibility  => expected rho NEGATIVE.
# H4: FEWER days since update (fresher)  => more visibility  => expected rho NEGATIVE.
HYP = [
    ("H1 Content Depth",    "word_count",             df["word_count"],              df[TARGET],        +1),
    ("H2 Search Demand",    "search_volume",          df["search_volume"],           df[TARGET],        +1),
    ("H2b log(1+volume)",   "log1p(search_volume)",   df["log_search_volume_tmp"],   df[TARGET],        +1),
    ("H3 SERP Position",    "avg_position (>0)",      ranked["avg_position"],        ranked[TARGET],    -1),
    ("H4 Content Freshness","days_since_last_update", df["days_since_last_update"],  df[TARGET],        -1),
]
rows = []
for hid, var, x, y, sign in HYP:
    rho, p_rho, r, p_r, n = corr_report(x, y)
    rows.append({"Hypothesis": hid, "Variable": var, "n": n, "Spearman_rho": rho, "p_spearman": p_rho,
                 "Pearson_r": r, "p_pearson": p_r, "Verdict": verdict(rho, p_rho, sign)})
hyp = pd.DataFrame(rows)
print("HYPOTHESIS VERDICT SUMMARY")
display(hyp.style.format({"Spearman_rho": "{:+.4f}", "Pearson_r": "{:+.4f}", "p_spearman": "{:.2e}", "p_pearson": "{:.2e}"}))

# ----------------------------------------------------------------------
# 3b. 2x2 audit chart
# ----------------------------------------------------------------------
H = hyp.set_index("Hypothesis")
samp = df.sample(min(6000, len(df)), random_state=RANDOM_STATE)
samp_r = samp[samp["avg_position"] > 0]

def panel(ax, x, y, xlabel, title, key):
    m_ = x.notna() & y.notna()          # NaNs (e.g. missing word_count / search_volume) cannot be plotted or fitted
    x, y = x[m_], y[m_]
    ax.scatter(x, y, s=6, alpha=0.2)
    m, b = np.polyfit(x, y, 1)
    xs = np.linspace(x.min(), x.max(), 100)
    ax.plot(xs, m * xs + b, color="crimson", lw=2)
    row = H.loc[key]
    ax.set(xlabel=xlabel, ylabel=TARGET, title=title)
    ax.text(0.03, 0.96, f"Verdict: {row['Verdict']}\nSpearman {row['Spearman_rho']:+.3f} | Pearson {row['Pearson_r']:+.3f}",
            transform=ax.transAxes, va="top", fontsize=10, bbox=dict(boxstyle="round", fc="white", alpha=0.85))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
panel(axes[0, 0], samp["word_count"], samp[TARGET], "word_count", "H1: Content Depth", "H1 Content Depth")
panel(axes[0, 1], samp["log_search_volume_tmp"], samp[TARGET], "log(1 + search_volume)", "H2: Search Demand", "H2b log(1+volume)")
panel(axes[1, 0], samp_r["avg_position"], samp_r[TARGET], "avg_position (ranked pages only)", "H3: SERP Positioning", "H3 SERP Position")
panel(axes[1, 1], samp["days_since_last_update"], samp[TARGET], "days_since_last_update", "H4: Content Freshness", "H4 Content Freshness")
plt.suptitle("Hypothesis Audit: Industry SEO Heuristics vs. Realized Visibility", fontsize=15, y=1.01)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "hypothesis_audit_distributions.png"), dpi=150, bbox_inches="tight")
plt.show()
df = df.drop(columns=["log_search_volume_tmp"])

### Interpretation of Hypotheses Findings
* **Guide for your write-up:**
  * Explain in plain words what each statistical verdict revealed about search traffic.
  * Why did H2 (search volume) show a mixed/weak correlation? (Broad head-terms face high competition).
  * Why was H3 (rank position) false when tested in isolation? (Position without search demand yields zero views).
  * Why did H4 (content freshness) show an opposite result? (Cumulative domain authority and backlink history outweigh recent date updates).

## 4. Feature Vector Assembly & Leakage Defense

### 4.1 Strict Leakage Blacklists
To ensure 100% scientific integrity, the feature pipeline strictly blacklists all leakers:
* **Pseudonymous Identifiers:** `content_id`, `client_id` (used solely for grouping and validation assertions).
* **Direct Target Telemetry:** `impressions_90d`, `clicks_90d`, `ctr` (since CTR contains impressions in its denominator).
* **Outcome Window Metrics:** `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d`.
* **Target-Derived Labels:** `trend_direction`, `trend_pct`, `impression_tier`.

### 4.2 Domain-Specific Feature Engineering
Engineer high-leverage structural and behavioral signals:
1. **SERP Power-Law Inverse Rank (`serp_rank_power`):**
   $$\text{serp\_rank\_power} = \begin{cases} \frac{1}{\text{avg\_position}}, & \text{if } \text{avg\_position} > 0 \\ 0, & \text{otherwise} \end{cases}$$
2. **Rank Presence Indicator:** `has_rank_data = (avg_position > 0).astype(int)`.
3. **Rank Imputation:** `imputed_avg_position = np.where(avg_position > 0, avg_position, 100.0)`.
4. **Behavioral Ratios:**
   * `engaged_session_ratio = engaged_sessions_90d / (sessions_90d + 1)`
   * `scroll_per_pageview = scroll_events_90d / (pageviews_90d + 1)`
   * `ai_session_ratio = ai_sessions_90d / (sessions_90d + 1)`
5. **Demand vs. Position Synergy:** `demand_position_synergy = log_search_volume * serp_rank_power`.
6. **Engagement vs. Volume Synergy:** `engagement_volume_synergy = log_sessions_90d * serp_rank_power`.
7. **Systematic Missingness Indicators:** `missing_word_count`, `missing_search_volume`, `missing_competition`, `missing_cpc`.
8. **Log-Stabilizing Transforms:** Applied to heavy-tailed count columns (`word_count`, `search_volume`, `sessions_90d`, etc.).
9. **One-Hot Categorical Encoding:** Applied to `content_type`, `main_intent`, `competition_level`, `freshness_tier`, `position_tier`.

In [ ]:
# Code Cell 4: Feature Vector Assembly and Dataset Export
# ----------------------------------------------------------------------
# 4a. LEAKAGE BLACKLIST (applied first, so nothing downstream can touch these columns)
# ----------------------------------------------------------------------
META = ["content_id", "client_id", TARGET]           # carried for grouping / reporting, NEVER used as features
LEAK_EXACT = [
    "impressions_90d", "clicks_90d", "traffic_90d", "conversions_90d", "ctr",       # direct target telemetry
    "trend_direction", "trend_pct", "impression_tier",                               # target-derived labels
    "health_score", "priority_score", "action_type",                                 # internal decision flags
    "url", "title", "keyword", "days_with_impressions",                              # free-text identifiers
]
LEAK_REGEX = r"(_last_30d|_prev_30d)$|^days_with_"                                               # outcome-window metrics
leak_cols = [c for c in df.columns if c not in META and
             (c in LEAK_EXACT or pd.Series(c).str.contains(LEAK_REGEX).iloc[0])]
print(f"Blacklisted columns present in data ({len(leak_cols)}):", leak_cols)

feat = df.drop(columns=leak_cols).copy()
has = lambda *cols: all(c in feat.columns for c in cols)

# ----------------------------------------------------------------------
# 4b. Systematic-missingness indicators (BEFORE any imputation)
# ----------------------------------------------------------------------
for c in ["word_count", "search_volume", "competition", "cpc"]:
    if c in feat.columns:
        feat[f"missing_{c}"] = feat[c].isna().astype(int)

# ----------------------------------------------------------------------
# 4c. SERP power-law features + the unranked trap (avg_position == 0 is NOT rank zero)
# ----------------------------------------------------------------------
ap = feat["avg_position"]
feat["has_rank_data"] = (ap > 0).astype(int)
feat["imputed_avg_position"] = np.where(ap > 0, ap, 100.0)
feat["serp_rank_power"] = np.where(ap > 0, 1.0 / ap.where(ap > 0, 1.0), 0.0)
feat["log_avg_position"] = np.log(feat["imputed_avg_position"])
feat = feat.drop(columns=["avg_position"])           # raw column is ambiguous (0 = unranked); replaced above

# ----------------------------------------------------------------------
# 4d. Rate columns are stored x100 -> convert to fractions
# ----------------------------------------------------------------------
for c in ["engagement_rate", "scroll_rate", "ai_traffic_pct"]:
    if c in feat.columns:
        feat[c] = feat[c] / 100.0

# ----------------------------------------------------------------------
# 4e. Behavioral ratios (built from RAW counts, before log transforms)
# ----------------------------------------------------------------------
if has("engaged_sessions_90d", "sessions_90d"):
    feat["engaged_session_ratio"] = feat["engaged_sessions_90d"] / (feat["sessions_90d"] + 1)
if has("scroll_events_90d", "pageviews_90d"):
    feat["scroll_per_pageview"] = feat["scroll_events_90d"] / (feat["pageviews_90d"] + 1)
if has("ai_sessions_90d", "sessions_90d"):
    feat["ai_session_ratio"] = feat["ai_sessions_90d"] / (feat["sessions_90d"] + 1)

# ----------------------------------------------------------------------
# 4f. Log-stabilizing transforms for heavy-tailed, non-negative counts
# ----------------------------------------------------------------------
EXPLICIT_LOG = ["word_count", "search_volume", "sessions_90d", "pageviews_90d", "internal_links_in",
                "external_backlinks", "content_age_days", "days_since_last_update", "cpc",
                "engaged_sessions_90d", "scroll_events_90d", "ai_sessions_90d"]
to_log = [c for c in EXPLICIT_LOG if c in feat.columns]

# Also auto-detect any other heavy-tailed count column so nothing skewed slips through
skip = set(META) | set(to_log) | {"serp_rank_power", "imputed_avg_position", "log_avg_position"}
for c in feat.select_dtypes("number").columns:
    if c in skip or c.startswith(("missing_", "has_")) or c.endswith(("_ratio", "_rate", "_pct")):
        continue
    s = feat[c].dropna()
    if len(s) and s.min() >= 0 and s.nunique() > 20 and s.skew() > 3:
        to_log.append(c)
print("log1p applied to:", to_log)

for c in to_log:
    feat[f"log_{c}"] = np.log1p(feat[c].clip(lower=0))
feat = feat.drop(columns=to_log)

# ----------------------------------------------------------------------
# 4g. Interaction synergy terms
# ----------------------------------------------------------------------
if has("log_search_volume"):
    feat["demand_position_synergy"] = feat["log_search_volume"].fillna(0) * feat["serp_rank_power"]
if has("log_sessions_90d"):
    feat["engagement_volume_synergy"] = feat["log_sessions_90d"] * feat["serp_rank_power"]

# ----------------------------------------------------------------------
# 4h. One-hot encode categoricals
# ----------------------------------------------------------------------
CAT_COLS = [c for c in ["content_type", "main_intent", "competition_level", "freshness_tier", "position_tier"]
            if c in feat.columns]
feat = pd.get_dummies(feat, columns=CAT_COLS, dtype=int)
print("One-hot encoded:", CAT_COLS)

# Any other non-numeric column cannot be modelled -> drop & report
other_obj = [c for c in feat.columns if c not in ("content_id", "client_id") and not pd.api.types.is_numeric_dtype(feat[c])]
if other_obj:
    print("Dropping unmodelled non-numeric columns:", other_obj)
    feat = feat.drop(columns=other_obj)

# ----------------------------------------------------------------------
# 4i. Impute remaining NaNs with the median (missing_* flags already preserve the 'was missing' signal)
#     NOTE: the median is computed on all rows; it is a single constant per column and a negligible leak.
# ----------------------------------------------------------------------
feature_cols = [c for c in feat.columns if c not in META]
na_cols = [c for c in feature_cols if feat[c].isna().any()]
if na_cols:
    print("Median-imputing:", na_cols)
    feat[na_cols] = feat[na_cols].fillna(feat[na_cols].median())

# ----------------------------------------------------------------------
# 4j. Integrity assertions (fail loudly on any leak / NaN)
# ----------------------------------------------------------------------
assert feat[feature_cols].isna().sum().sum() == 0, "NaNs remain in feature matrix"
assert np.isfinite(feat[feature_cols].to_numpy(dtype=float)).all(), "Inf values in feature matrix"
assert not (set(feature_cols) & set(LEAK_EXACT)), "Blacklisted column leaked into features!"
assert not [c for c in feature_cols if pd.Series(c).str.contains(LEAK_REGEX).iloc[0]], "30-day window column leaked!"
assert not (set(feature_cols) & {"content_id", "client_id", TARGET}), "ID/target in feature vector!"
bad = [c for c in feature_cols if any(k in c.lower() for k in ("impression", "click"))]
assert not bad, f"Suspicious outcome-like feature names: {bad}"
assert len(feat) == len(df_raw)

# ----------------------------------------------------------------------
# 4k. Export (parquet if an engine is installed, plus CSV as the template requires)
# ----------------------------------------------------------------------
csv_path = os.path.join(OUT_DIR, "prepared_features.csv")
feat.to_csv(csv_path, index=False)
try:
    feat.to_parquet(os.path.join(OUT_DIR, "prepared_features.parquet"), index=False)
    print("Saved parquet + csv")
except Exception as e:
    print(f"Parquet engine unavailable ({type(e).__name__}); CSV only -> Notebook 02 falls back to the CSV.")

with open(os.path.join(OUT_DIR, "feature_list.json"), "w") as f:
    json.dump(feature_cols, f, indent=2)

print(f"\nPrepared matrix: {feat.shape[0]:,} rows x {len(feature_cols)} features (+3 meta columns)")
print("Zero target leakers confirmed. Features:\n", feature_cols)